# Step 3: SageMaker Fine-tuning 실행

SageMaker Training Job을 사용하여 한국인 딥페이크 데이터(KoDF)로 모델을 Fine-tuning합니다.

## 실습 목표
- SageMaker PyTorch Estimator 구성
- Training Job 실행
- 학습 모니터링

## 3.1 환경 설정

In [ ]:
import json
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker import get_execution_role

# 설정 로드
with open('../config.json', 'r') as f:
    config = json.load(f)

# SageMaker 세션
sagemaker_session = sagemaker.Session()
role = config['role']
bucket = config['bucket']
prefix = config['prefix']

print(f"Role: {role}")
print(f"Bucket: {bucket}")

## 3.2 하이퍼파라미터 설정

In [ ]:
# 100-200 레벨에 맞춘 간단한 하이퍼파라미터
hyperparameters = {
    'epochs': 5,              # 빠른 실습을 위해 5 에포크
    'batch-size': 32,
    'learning-rate': 0.0001,
    'model-name': 'efficientnet_b0'
}

print("하이퍼파라미터:")
for k, v in hyperparameters.items():
    print(f"  {k}: {v}")

## 3.3 PyTorch Estimator 생성

In [ ]:
# SageMaker PyTorch Estimator
estimator = PyTorch(
    entry_point='train.py',
    source_dir='.',
    role=role,
    instance_count=1,
    instance_type='ml.g4dn.xlarge',  # GPU 인스턴스
    framework_version='2.0.0',
    py_version='py310',
    hyperparameters=hyperparameters,
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=sagemaker_session,
    # 비용 절감을 위한 Spot 인스턴스 (선택사항)
    # use_spot_instances=True,
    # max_wait=7200,
    # max_run=3600,
)

print("PyTorch Estimator 생성 완료")
print(f"Instance Type: ml.g4dn.xlarge")
print(f"Framework: PyTorch 2.0.0")

## 3.4 Training Job 실행

In [ ]:
# 데이터 채널 설정
data_channels = {
    'train': config['s3_train_path'],
    'val': config['s3_val_path']
}

print("데이터 채널:")
for k, v in data_channels.items():
    print(f"  {k}: {v}")

In [ ]:
# Training Job 시작
print("=" * 60)
print("  SageMaker Training Job 시작")
print("  한국인 딥페이크 데이터로 Fine-tuning 중...")
print("=" * 60)

estimator.fit(data_channels, wait=True, logs='All')

## 3.5 학습 결과 확인

In [ ]:
# 학습된 모델 경로
model_data = estimator.model_data
print(f"학습된 모델 위치: {model_data}")

# Training Job 이름 저장
training_job_name = estimator.latest_training_job.name
print(f"Training Job 이름: {training_job_name}")

# 설정 업데이트
config['model_data'] = model_data
config['training_job_name'] = training_job_name

with open('../config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("설정이 업데이트되었습니다.")

In [ ]:
# 학습 메트릭 확인 (CloudWatch)
from sagemaker.analytics import TrainingJobAnalytics

try:
    training_analytics = TrainingJobAnalytics(training_job_name)
    df = training_analytics.dataframe()
    print("학습 메트릭:")
    display(df)
except Exception as e:
    print(f"메트릭 조회 실패 (정상일 수 있음): {e}")

## 완료!

SageMaker Fine-tuning이 완료되었습니다.

**결과:**
- 한국인 딥페이크 데이터(KoDF)로 모델이 Fine-tuning됨
- 학습된 모델이 S3에 저장됨

**➡️ 다음 단계: `4_after_evaluation/evaluate_after.ipynb`**

Fine-tuned 모델의 성능을 평가해봅니다!